# Reproduce Table 1 from the released NPZ

CPU-only analysis: no GPT-2 model or GPU is required. The notebook loads the released Phase II patch measurements, aggregates the eight trajectory positions to 4,800 pair-level observations, computes prompt-cluster bootstrap 95% CIs, and reproduces the recovery-vs-layer figure.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path('../data/phaseII_full_20prompts.npz')
if not DATA.exists(): DATA = Path('data/phaseII_full_20prompts.npz')
rows = np.load(DATA, allow_pickle=True)['rows']
df = pd.DataFrame(list(rows))
print(f'{len(df):,} patch measurements')
assert len(df) == 38400


In [ ]:
keys=['pair_id','prompt_idx','seed','rho','direction','layer']
pair=(df.groupby(keys,as_index=False).agg(recovery=('recovery','mean'), n=('t','count')))
assert len(pair)==4800 and set(pair.n)=={8}
print(f'{len(pair):,} pair-level observations')


In [ ]:
def prompt_cluster_bootstrap(sub, n_boot=5000, seed=20260912):
    rng=np.random.default_rng(seed)
    means=sub.groupby('prompt_idx')['recovery'].mean().to_numpy()
    draws=rng.choice(means,size=(n_boot,len(means)),replace=True).mean(axis=1)
    return means.mean(), *np.quantile(draws,[.025,.975])

records=[]
for direction in ['AR_to_IID','IID_to_AR']:
    for layer in [8,9,10]:
        sub=pair[(pair.direction==direction)&(pair.layer==layer)]
        m,lo,hi=prompt_cluster_bootstrap(sub)
        records.append(dict(direction=direction,layer=layer,mean_recovery=m,ci_low=lo,ci_high=hi,n_pair=len(sub),n_prompts=sub.prompt_idx.nunique()))
table=pd.DataFrame(records)
print(table.to_string(index=False,float_format=lambda x:f'{x:.3f}'))
table.to_csv('../results/table1_reproduced.csv',index=False)


In [ ]:
fig,ax=plt.subplots(figsize=(6.2,4.2))
markers={'AR_to_IID':'o','IID_to_AR':'s'}
for direction in ['AR_to_IID','IID_to_AR']:
    t=table[table.direction==direction].sort_values('layer')
    ax.errorbar(t.layer,t.mean_recovery,yerr=[t.mean_recovery-t.ci_low,t.ci_high-t.mean_recovery],marker=markers[direction],capsize=4,linewidth=1.5,label=direction.replace('_',' → '))
ax.set(xlabel='Residual-stream layer',ylabel='Mean recovery',xticks=[8,9,10],ylim=(.90,1.005))
ax.grid(alpha=.25); ax.legend(frameon=False); fig.tight_layout()
out=Path('../figures/recovery_vs_layer_bootstrap.png'); out.parent.mkdir(parents=True,exist_ok=True); fig.savefig(out,dpi=220,bbox_inches='tight'); plt.show()
